In [ ]:
import random
import math
import time

import pandas as pd
from pandas import DataFrame
from sklearn.neighbors import BallTree
import numpy as np

from geopy.distance import geodesic

startPoint = {"lat": 59.927085,"lon": 30.317504}
endPoint = {"lat": 59.935408, "lon": 30.327108}

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 1
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 10

random.seed(41)
np.random.seed(41)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [252]:
df = pd.read_csv('data/places.csv')

**Алгоритмы генетики**

In [253]:
# Генерация случайной особи

def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    buffer_km = 0.5

    # Переводим километры в градусы
    lat_buffer = buffer_km / 111

    mean_lat = (startPoint["lat"] + endPoint["lat"]) / 2
    lon_buffer = buffer_km / (111 * np.cos(np.radians(mean_lat)))

    min_lat = min(startPoint["lat"], endPoint["lat"]) - lat_buffer
    max_lat = max(startPoint["lat"], endPoint["lat"]) + lat_buffer

    min_lon = min(startPoint["lon"], endPoint["lon"]) - lon_buffer
    max_lon = max(startPoint["lon"], endPoint["lon"]) + lon_buffer

    df_filtered = df[
        (df["lat"] >= min_lat) &
        (df["lat"] <= max_lat) &
        (df["lon"] >= min_lon) &
        (df["lon"] <= max_lon)
    ]

    population = []

    df_indexes = df_filtered["id"].tolist()

    for _ in range(num):
        route_id = random.sample(
            range(len(df_indexes)),
            route_size
        )
        route = []
        for i in route_id:
            route.append(df_indexes[i])
        population.append({"route": route, "fitness": None})

    return population

In [254]:
population = create_population(df, startPoint, endPoint, POPULATION_SIZE)


In [255]:
import folium

individual = population[0]
route_df = df[(df["id"].isin(individual["route"]))]

m = folium.Map(
    location=[
        route_df["lat"].mean(),
        route_df["lon"].mean()
    ],
    zoom_start=15
)

# Точки маршрута
folium.Marker(
    [startPoint["lat"], startPoint["lon"]]
).add_to(m)
for i, (_, row) in enumerate(route_df.iterrows()):
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"{i+1}. {row['name']}"
    ).add_to(m)
folium.Marker(
    [endPoint["lat"], endPoint["lon"]]
).add_to(m)

# Линия маршрута
folium.PolyLine(
    route_df[["lat", "lon"]].values,
    weight=4
).add_to(m)

m.save(r"visualisations\route.html")